# Phase 1: NLP Feature Engineering

Machine learning models cannot understand raw text. They require numerical input.
Thus, we transform text into mathematical representations that models can process.

In this phase, we:

- Convert raw customer reviews into numerical representations (text vectorization)
- Explore Bag of Words (BoW) and TF-IDF techniques
- Apply preprocessing techniques such as stopword removal and lemmatization
- Construct a clean and structured feature matrix for machine learning models

## 1. Data Loading 

Load the cleaned dataset prepared in Phase 0.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_reviews.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (157432, 2)


,Cleaned_Text,Sentiment
0,this is perfetly identical to jiffy peanutbutt...,Negative
1,they taste more like chemicals than meat i lik...,Negative
2,chips were ok just didn t have enough dill and...,Negative
3,very disappointed to receive a case of food pr...,Negative
4,i absolutely hate the taste of this tea but if...,Negative


## 2. Train-Test Split

We split the dataset into training and testing sets using stratified sampling to preserve class distribution.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

X = df['Cleaned_Text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", X_train.shape)
print("Test size:", X_test.shape)

Training size: (125945,)
Test size: (31487,)


## 3. Text Preprocessing

We apply:
- Stopword removal to eliminate common non-informative words
- Lemmatization to normalize words to their base form

Preprocessing is applied after train-test split to prevent data leakage.

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.preprocessing import preprocess_text
from src.feature_engineering import get_tfidf_vectorizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Apply preprocessing
X_train_clean = X_train.apply(preprocess_text)
X_test_clean = X_test.apply(preprocess_text)

X_train_clean.head()

23690     buy local grocery store want get iodine need d...
66761     one family favorite little indulgences except ...
19070     waste money lemonade taste like waste water bi...
5047      cereal absolutely fantastic great review subsc...
101307    easy order recevied oredered faster think good...
Name: Cleaned_Text, dtype: object

## 4. N-grams for Context Capture

Single words (unigrams) may fail to capture contextual meaning in text.

For example:
- "not good" (negative)
- "very good" (positive)

To address this, we use n-grams, which consider sequences of words.

In this project, we use:
- Unigrams (single words)
- Bigrams (two-word combinations)

This allows the model to capture phrase-level sentiment information.

## 5. TF-IDF Feature Engineering

We convert text into numerical vectors using TF-IDF with n-gram support.

Key configurations:
- max_features: limits vocabulary size
- ngram_range=(1,2): includes unigrams and bigrams
- min_df: removes rare words
- max_df: removes overly common words

The vectorizer is fit only on training data to avoid data leakage.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = get_tfidf_vectorizer()

# Fit on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)

# Transform test data
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (125945, 5000)
Test TF-IDF shape: (31487, 5000)


## 6. Feature Insights

We inspect the learned vocabulary to understand the feature space.

In [5]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total features:", len(feature_names))
print("Sample features:", feature_names[:30])

Total features: 5000
Sample features: ['ability' 'able' 'able buy' 'able find' 'able get' 'absolute'
 'absolute favorite' 'absolutely' 'absolutely delicious' 'absolutely love'
 'absolutely no' 'absorb' 'acai' 'accept' 'acceptable' 'access' 'accord'
 'account' 'accurate' 'accustom' 'ache' 'acid' 'acidic' 'acidity' 'acids'
 'acquire' 'acquire taste' 'across' 'act' 'active']


## 7. Sparsity Analysis

TF-IDF matrices are highly sparse, meaning most values are zero.

In [6]:
non_zero = X_train_tfidf.nnz
total = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]

sparsity = 1 - (non_zero / total)

print("Non-zero values:", non_zero)
print("Total values:", total)
print("Sparsity:", sparsity)

Non-zero values: 4399240
Total values: 629725000
Sparsity: 0.9930140299337012


## 8. Save Processed Features

To ensure reproducibility and modularity, we save:

- TF-IDF feature matrices
- Target labels

In [7]:
import joblib
import os

# Save features
joblib.dump(X_train_tfidf, "../data/processed/X_train_tfidf.pkl")
joblib.dump(X_test_tfidf, "../data/processed/X_test_tfidf.pkl")

# Save labels
joblib.dump(y_train, "../data/processed/y_train.pkl")
joblib.dump(y_test, "../data/processed/y_test.pkl")

# (VERY IMPORTANT) Save vectorizer
joblib.dump(tfidf_vectorizer, "../data/processed/tfidf_vectorizer.pkl")

print("All processed data saved successfully!")

All processed data saved successfully!


## 9. Final Feature Matrix

We now have:

- X_train_tfidf
- X_test_tfidf

Each review is represented as a 5000-dimensional vector.

This feature matrix is ready for machine learning models.